In [ ]:
# Combined sc and sn scVI integration restricted to clean HVGs.
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

In [ ]:
input_dir = "/home/yanjun3/atlas/heart_sc"

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/atlas_project/heart_sc_old/qc3_scvi.h5ad')
adata1 = sc.read('/home/yanjun3/atlas/heart_sn/sn_final.h5ad')

In [ ]:
adata

In [ ]:
adata.obs["anatomical_site_prev"] = adata.obs["anatomical_site"]

adata.obs["anatomical_site"] = (
    adata.obs["anatomical_site_prev"].astype(str)
    + "_"
    + adata.obs["rna_source"].astype(str)
)

In [ ]:
adata1.obs["anatomical_site_prev"] = adata1.obs["anatomical_site"]

adata1.obs["anatomical_site"] = (
    adata1.obs["anatomical_site_prev"].astype(str)
    + "_"
    + adata1.obs["rna_source"].astype(str)
)

In [ ]:
adata = adata.concatenate(adata1, join = 'outer')

In [ ]:
adata.obs_names_make_unique()

In [ ]:
del adata1

In [ ]:
import scvi
import seaborn as sns
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker

In [ ]:
sc.set_figure_params(figsize=(6, 6),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sc.settings.figdir = "/home/yanjun3/atlas/heart_sc/scsn_cleanhvg"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
batch_key = "sample"

In [ ]:
#is_ribo = adata.var_names.str.startswith(("RPL", "RPS"))
#is_mt = adata.var_names.str.startswith("MT-")
#adata.var["mt"] = adata.var_names.str.startswith("MT-")
#adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
#adata_clean = adata[:, ~(is_ribo | is_mt)].copy()
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo"], inplace=True, percent_top=[20], log1p=True
)
adata

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=8000)
adata.var.loc[adata.var_names.str.startswith(("MT-", "RPL", "RPS")),
              "highly_variable"] = False

In [ ]:
adata.var.highly_variable.sum()

In [ ]:
import random
random.seed(42)
print(random.random()) 

In [ ]:
genes_keep = ~adata.var_names.str.startswith(("MT-", "RPL", "RPS"))
adata_scvi = adata[:, genes_keep].copy()

In [ ]:
# scVI integration
scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer="counts",
    batch_key=batch_key,
    continuous_covariate_keys=[
        "pct_counts_mt",
        "n_genes_by_counts",
        "total_counts"
    ]
)

In [ ]:
model_scvi = scvi.model.SCVI(adata_scvi)
model_scvi

In [ ]:
model_scvi.view_anndata_setup()

In [ ]:
import numpy as np
max_epochs_scvi = np.min([round((20000 / adata_scvi.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
model_scvi.train()

In [ ]:
adata.obsm["X_scVI"] = model_scvi.get_latent_representation()

In [ ]:
adata

In [ ]:
adata.var = adata.var.loc[:, ~adata.var.columns.str.contains(r"-\d+$")]
#adata.write("snsc_scvi.h5ad")


In [ ]:
adata

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=["anatomical_site_prev", "anatomical_site", "rna_source"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'right margin',
    #save = '0_3_v1.png'
)

In [ ]:
sc.tl.leiden(adata, key_added="leiden_res0_3", resolution=0.3)
sc.pl.umap(
    adata,
    color=["leiden_res0_3"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data',
    save = '0_3_v1.png'
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
obs_df = adata.obs[['rna_source', 'anatomical_site_prev']]
frequency_df = obs_df.groupby(['rna_source', 'anatomical_site_prev']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of anatomical_site in rna_source")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()


In [ ]:
obs_df = adata.obs[['rna_source', 'anatomical_site']]
frequency_df = obs_df.groupby(['rna_source', 'anatomical_site']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of anatomical_site in rna_source")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()


In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res0_3", "rna_source", "anatomical_site_prev"]
    ,
    frameon=False,
    ncols=2, size = 30, legend_loc = 'right margin',
   # save = '0_3_v1.png'
)

In [ ]:
obs_df = adata.obs[['rna_source', 'leiden_res0_3']]
frequency_df = obs_df.groupby(['rna_source', 'leiden_res0_3']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of leiden_res0_3 in rna_source")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()

In [ ]:
sc.tl.embedding_density(adata, basis='umap', groupby='rna_source')
sc.pl.embedding_density(adata, basis='umap', key='umap_density_rna_source')

In [ ]:
sc.tl.embedding_density(adata, basis='umap', groupby='anatomical_site_prev')
sc.pl.embedding_density(adata, basis='umap', key='umap_density_anatomical_site_prev')

In [ ]:
sc.tl.dendrogram(adata, groupby="leiden_res0_3")

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata, groupby="leiden_res0_3", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, standard_scale='var', 
                             #   save = 'snsc_res_0_3_topmarkers'
                               )

In [ ]:
# Save result
adata.write('cleanhvg.h5ad')

In [ ]:
!pwd

In [ ]:
sc.pl.rank_genes_groups_heatmap(adata, groupby="leiden_res0_3", n_genes = 20, show_gene_labels=True, cmap="RdBu_r", dendrogram = True,
                               vmin = 0, vmax = 4, vcenter = 2, swap_axes = True, save = 'snsc_leiden_res0_3.png')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv(f"{input_dir}/snsc_res03.csv", index=False)

In [ ]:
sc.pl.dotplot(
    adata, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
              'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA', 
               'NRP1', 'FMN1', 'DAAM2', 'AP2A2', 'DNM1', 'CARMIL1',
              'SPP1', 'APOE', 'CLEC10A', 'HDAC9',
              'FCGR2A', 'FCGR2B',
              'CLSPN', 'STIL'],
    groupby='leiden_res0_3',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
 #  save = 'scvi_08.png'
)

In [ ]:
adata.write('snsc_final.h5ad')

In [ ]:
import numpy as np
import pandas as pd

X = adata.layers["counts"]
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)

expr = pd.DataFrame(X.T, index=adata.var_names, columns=adata.obs_names)
expr.to_csv("expr.txt", sep="\t")


In [ ]:
del sc_adata

In [ ]:
del sn_adata

In [ ]:
del adata_sub

In [ ]:
from cytotrace2_py.cytotrace2_py import cytotrace2

res = cytotrace2("expr.txt", species="human")

In [ ]:
scores = res["results"][["cell_id", "cytotrace2_score"]].set_index("cell_id").reindex(adata.obs_names)
adata.obs["cytotrace2_score"] = scores["cytotrace2_score"].values 

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)


In [ ]:

marker_dict = {
    "cd14 monocytes": [
        "S100A9", "S100A8", "LYZ", "VCAN", "NAMPT", "S100A4",
        "TYROBP", "SRGN", "FCER1G", "PLAUR", "CTSS", "S100A12",
        "COTL1", "SAMSN1", "KYNU", "AIF1", "SAT1", "FTL", "FCN1",
    ],

    "cd16 monocytes": [
        "FCER1G", "TYROBP", "S100A9", "AIF1", "SAT1", "LST1",
        "FTL", "COTL1", "CTSS", "LYZ", "S100A4", "S100A8",
        "NAMPT", "FCN1", "FTH1", "HLA-DRA", "CYBA", "SRGN",
        "FCGR3A",
    ],

    "cDC1s": [
        "HLA-DRA", "HLA-DPB1", "HLA-DPA1", "CD74", "LYZ",
        "HLA-DQA1", "HLA-DRB1", "CCSER1", "HLA-DQB1", "CPVL",
        "HDAC9", "WDFY4", "IDO1", "BASP1", "CST3", "HLA-DRB5",
        "SAT1", "PTPRC", "CLNK",
    ],

    "cDC2s": [
        "HLA-DRA", "HLA-DPB1", "HLA-DRB1", "HLA-DPA1", "CD74",
        "HLA-DQA1", "HLA-DQB1", "LYZ", "TYROBP", "FCER1G",
        "SRGN", "AIF1", "MS4A6A", "CST3", "CD83", "FCER1A",
        "HLA-DMA", "SIPA1L1", "AREG",
    ],

    "cycling myeloid": [
        "C1QB", "LYZ", "C1QA", "S100A4", "CD74", "HLA-DRA",
        "C1QC", "TUBA1B", "AIF1", "STMN1", "MS4A6A", "HMGB2",
        "FCER1G", "CTSB", "CD163", "TYROBP", "HLA-DRB1",
        "F13A1", "RBPJ",
    ],

    "lyve1 macrophages": [
        "C1QA", "HLA-DRA", "C1QB", "CD74", "TYROBP", "C1QC",
        "FTL", "MS4A6A", "CD163", "F13A1", "CTSB", "AIF1",
        "HLA-DRB1", "RBPJ", "MRC1", "FCER1G", "LYZ",
        "HLA-DPB1", "RBM47",
    ],

    "spp1 macrophages": [
        "FTL", "CTSB", "TYROBP", "FTH1", "NAMPT", "PLAUR",
        "FCER1G", "CXCL8", "SLC16A10", "KYNU", "CD44", "LYZ",
        "S100A4", "SIPA1L1", "CXCL3", "LAPTM5", "CD163",
        "HLA-DRA", "CTSD",
    ],

    "TMEM163 macrophages": [
        "HLA-DRA", "CD74", "HLA-DPA1", "HLA-DPB1", "HLA-DRB1",
        "HLA-DQA1", "TYROBP", "C1QA", "C1QB", "C1QC",
        "HLA-DQB1", "CCL3", "CD83", "CCL4", "IER3", "SAT1",
        "CCL3L1", "FCGR2A", "AIF1", "FTL",
    ],
}

In [ ]:
sc.pl.dotplot(
    adata,
    var_names = marker_dict,
    groupby   = "leiden_res0_3",   # change to your obs column
    dendrogram = False,
    save = 'disco_fine.png'
)

In [ ]:
sc.pl.dotplot(
    adata, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
              'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA', 
              'SPP1', 'APOE',
              'FCGR2A', 'FCGR2B',
              'CLSPN', 'STIL'],
    groupby='leiden_res0_3',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
 #  save = 'scvi_08.png'
)

In [ ]:
sc.pl.umap(
    adata,
    color=["LYVE1", "F13A1", "MRC1", "leiden_res0_5"]
    ,
    frameon=False,
    ncols=2, size = 30, legend_loc = 'on data',
    #save = '0_3_v1.png'
)

In [ ]:
clusters = ["4", "7"]
cluster_key = "leiden_res0_3"

adata_sub = adata[adata.obs[cluster_key].isin(clusters)].copy()


In [ ]:
clusters = ["4","5", "9"]         
cluster_key = "leiden_res0_5"            
# cells in those clusters (from adata_hvg)
sel_cells = adata_hvg.obs_names[adata_hvg.obs[cluster_key].astype(str).isin(clusters)]

# mask onto your full adata, then subset
adata.obs["mask"] = adata.obs_names.isin(sel_cells)
adata_subset = adata[adata.obs["mask"]].copy()
adata_subset.write('group3_myeloid.h5ad')

In [ ]:
marker_dict = {
    # ---- Myeloid focus ----
    "Classical Monocytes": ["S100A8", "S100A9", "VCAN", "LYZ", "LST1", "CTSS", "CD14"],
    "Nonclassical Monocytes": ["FCGR3A", "MS4A7", "LILRB1"],
    "Intermediate Monocytes": ["CD14", "FCGR3A", "MS4A7"],
    "Macrophages (TRM core)": ["C1QA", "C1QB", "C1QC", "APOE", "MRC1", "CD163", "MSR1", "CSF1R", "ITGAM"],
    "LYVE1+ TRM (cardiac)": ["LYVE1", "F13A1", "PDGFB"],
    "SPP1+ Macrophages": ["SPP1", "FABP5", "FN1"],
    "Cycling Myeloid": ["MKI67", "TOP2A"],
    "Dendritic Cells": ["HLA-DRA", "ITGAX"],
    "Neutrophils": ["S100A8", "S100A9"],

    # ---- Lymphoid (to exclude) ----
    "T Cells (CD4+)": ["CD3E", "IL7R"],
    "T Cells (CD8+)/NK": ["CD8A", "NKG7", "GNLY", "PRF1"],
    "B Cells": ["CD74", "MS4A7"],

    # ---- Cardiac parenchyma ----
    "Cardiomyocytes": ["TNNI3", "MYH7", "ACTC1", "MYL2"],
    "Endothelial Cells": ["PECAM1", "VWF", "KDR", "ESAM", "ENG"],
    "Fibroblasts": ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA", "TAGLN"],
    "Smooth Muscle": ["ACTA2", "MYH11", "TAGLN"],
    "Pericytes": ["RGS5", "PDGFRB", "ABCC9"],
    "Epicardial/Mesothelial": ["WT1"],
    "Adipocytes (Cardiac/Pericardial)": ["FABP4"],
    "Erythroid": ["HBB", "HBA1"]
}


sc.pl.dotplot(
    adata_hvg, 
    var_names=marker_dict,
    groupby='leiden_res0_5',
    dot_max = 0.4,
    standard_scale='var', 
    color_map='RdBu_r' 
    
  # save = 'scvi_08.png'
)

In [ ]:
sc.pl.dotplot(
    adata_mac_hvg, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2'],
    groupby='leiden_res0_8',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
   save = 'scvi_08.png'
)\sc.pl.umap(
    adata_mac_hvg,
    color=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2', 
          'HLA-DQA1', 'HLA-DQB1', 'leiden_res0_8'],
    frameon=False,
    ncols=3, size = 30, legend_loc = 'on data', color_map = 'RdBu_r',
    save = 'tmp_umap'
)

In [ ]:
sc.pl.umap(
    adata_mac_hvg,
    color=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2', 
          'HLA-DQA1', 'HLA-DQB1', 'leiden_res0_8'],
    frameon=False,
    ncols=3, size = 30, legend_loc = 'on data', color_map = 'RdBu_r',
    save = 'tmp_umap'
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp

adata_obj = adata_mac_hvg          # your object
grp      = "leiden_res0_8"
targets  = {"0","3","5","11","14"}
combo    = "c0_3_5_11_14_vs_rest"
de_key   = "deg_combo"
gene_sets= "MSigDB_Hallmark_2020"

# 1) make two-level categorical label: target vs other
mask = adata_obj.obs[grp].astype(str).isin(targets)
adata_obj.obs[combo] = pd.Categorical(
    np.where(mask, "target", "other"),
    categories=["other", "target"]   # ensures both levels exist
)

# 2) DE: target vs other
sc.tl.rank_genes_groups(
    adata_obj, groupby=combo, groups=["target"], reference="other",
    method="wilcoxon", use_raw=False, key_added=de_key
)

# 3) build preranked list and run GSEA (tutorial style)
deg = sc.get.rank_genes_groups_df(adata_obj, group="target", key=de_key)
rnk = (deg[["names","logfoldchanges"]]
       .dropna().drop_duplicates("names")
       .set_index("names")["logfoldchanges"]
       .sort_values(ascending=False))

pre = gp.prerank(
    rnk=rnk, gene_sets=gene_sets, outdir=None,
    min_size=10, max_size=5000, permutation_num=1000,
    seed=42, threads=4          # use threads (processes is deprecated)
)

res = pre.res2d.copy()          # has columns like Term, NES, FDR q-val
res_up = res[res["NES"] > 0].sort_values("NES", ascending=False)

# (optional) quick plots
gp.barplot(res_up, column="NES", title=f"Upregulated terms: {combo}")
gp.dotplot(res_up, column="FDR q-val", title=f"Upregulated terms: {combo}")


In [ ]:
import scanpy as sc
import gseapy as gp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- inputs you set ----
adata_obj   = adata_mac_hvg          # your AnnData (HVG or all-genes)
group_var   = "leiden_res0_8"        # cluster column
target_lab  = "3"                    # e.g., cluster 3 (or "target" if you merged clusters)
de_key      = "deg_for_enrichr"      # where DE is stored
libraries   = ["MSigDB_Hallmark_2020", "GO_Biological_Process_2021", "Reactome_2016"]
organism    = "Human"                # "Mouse" if mouse symbols

# 1) DE: target vs rest (wilcoxon)
sc.tl.rank_genes_groups(
    adata_obj, groupby=group_var, groups=[target_lab], reference="rest",
    method="wilcoxon", use_raw=False, key_added=de_key
)

# 2) Build UP/DOWN gene lists (adjust thresholds as needed)
deg = sc.get.rank_genes_groups_df(adata_obj, group=target_lab, key=de_key).dropna()
padj_thr, lfc_thr, top_n = 0.05, 0.25, 200

deg_up = (deg.query("pvals_adj < @padj_thr and logfoldchanges >  @lfc_thr")
            .sort_values(["pvals_adj","logfoldchanges"], ascending=[True, False]))
deg_dn = (deg.query("pvals_adj < @padj_thr and logfoldchanges < -@lfc_thr")
            .sort_values(["pvals_adj","logfoldchanges"], ascending=[True, True]))

up_genes = deg_up["names"].drop_duplicates().head(top_n).tolist()
dn_genes = deg_dn["names"].drop_duplicates().head(top_n).tolist()

print(f"UP genes: {len(up_genes)}, DOWN genes: {len(dn_genes)}")

# 3) Enrichr ORA (GSEApy)
enr_up = gp.enrichr(
    gene_list=up_genes,
    gene_sets=libraries,
    organism=organism,
    outdir=None, # keep in-memory
    cutoff=1.0   # don't filter; we'll sort ourselves
)
enr_dn = gp.enrichr(
    gene_list=dn_genes,
    gene_sets=libraries,
    organism=organism,
    outdir=None,
    cutoff=1.0
)

# 4) Quick view: top terms per library (UP)
def show_top(enr_res, title, n=10):
    df = enr_res.res2d.copy()
    # unify column names a bit
    df.columns = [c.replace("_", " ") for c in df.columns]
    keep = df.rename(columns={
        "Adjusted P-value":"Adjusted P-value",
        "Combined Score":"Combined Score",
        "Term":"Term",
        "Gene_set":"Gene set"
    })
    print(f"\n{title}")
    for lib in keep["Gene set"].unique():
        sub = keep[keep["Gene set"]==lib].sort_values("Adjusted P-value").head(n)
        print(f"\n— {lib}")
        print(sub[["Term","Adjusted P-value","Combined Score"]].to_string(index=False))

show_top(enr_up, f"UP in {group_var}={target_lab}")
show_top(enr_dn, f"DOWN in {group_var}={target_lab}")

# 5) Tutorial-style plots
# Barplot and dotplot for UP (filter one library or stack all)
up_df = enr_up.res2d.copy()
lib_focus = "MSigDB_Hallmark_2020"   # change per plot
up_df = up_df[up_df["Gene_set"]==lib_focus].sort_values("Adjusted P-value").head(15)

gp.barplot(
    up_df,
    title=f"Enrichr (UP): {group_var}={target_lab} — {lib_focus}",
    column="Adjusted P-value"
)
gp.dotplot(
    up_df,
    title=f"Enrichr (UP): {group_var}={target_lab} — {lib_focus}",
    column="Adjusted P-value",
    cmap=None, size=6
)

# (Optional) same plots for DOWN
dn_df = enr_dn.res2d.copy()
dn_df = dn_df[dn_df["Gene_set"]==lib_focus].sort_values("Adjusted P-value").head(15)
gp.barplot(dn_df, title=f"Enrichr (DOWN): {group_var}={target_lab} — {lib_focus}",
           column="Adjusted P-value")
gp.dotplot(dn_df, title=f"Enrichr (DOWN): {group_var}={target_lab} — {lib_focus}",
           column="Adjusted P-value", cmap=None, size=6)


In [ ]:
adata_mac_hvg.obs

In [ ]:
obs_df = adata_mac_hvg.obs[['Disease', 'leiden_res0_8']]
frequency_df = obs_df.groupby(['Disease', 'leiden_res0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters in obese / lean")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("samplefreq.pdf", dpi=600)
plt.show()
